# Polymarket BTC Up/Down 5m order book: quick start

This notebook works through 14 consecutive days (2026-08-08 to 2026-08-21 UTC) of tick-level Polymarket data: every L2
order book update, trade and on-chain fill for all 4,032 BTC Up/Down 5-minute markets, plus the Chainlink BTC/USD 60s
TWAP they resolve on.

You will:
1. load the market list and outcomes,
2. measure collection latency from the two timestamps on every row,
3. plot one market's best bid and ask and read a full-depth snapshot,
4. check every market's outcome against the Chainlink TWAP, day by day.

Data: [PMData](https://pmdata.dev) · [GitHub](https://github.com/mileswangs/polymarket-historical-data)

In [ ]:
import glob, os
import polars as pl
import matplotlib.pyplot as plt

DATA = os.environ.get("PMDATA_SAMPLE_DIR") or os.path.dirname(glob.glob("/kaggle/input/**/markets.parquet", recursive=True)[0])
print(DATA)

## 1. Markets
One row per market, 288 a day. `winning_outcome` comes from Polymarket's `market_resolved` message: `yes` = Up, `no` = Down.

In [ ]:
markets = pl.read_parquet(f"{DATA}/markets.parquet")
print(markets.group_by("winning_outcome").len())
markets.head()

## 2. Collection latency
Every row carries Polymarket's event time (`timestamp`) and PMData's receive time (`local_timestamp`).
The difference is collection latency. Measure it on `price_change` rows.

In [ ]:
l2 = pl.scan_parquet(f"{DATA}/l2/*/*.parquet")
lat = (
    l2.filter(pl.col("event_type") == "price_change")
    .select(((pl.col("local_timestamp") - pl.col("timestamp")).dt.total_microseconds() / 1000).alias("latency_ms"))
    .collect()
)
print(lat.select(
    pl.col("latency_ms").quantile(0.5).alias("p50_ms"),
    pl.col("latency_ms").quantile(0.9).alias("p90_ms"),
    pl.len().alias("events"),
))

## 3. One market's order book
Every field comes straight from Polymarket's WebSocket messages. `price_change` rows carry Polymarket's best bid
and ask after each change; `book` rows are full snapshots of every level. Files start a few minutes before the
market opens, so cut to the 5 minutes that start at the slug timestamp.

In [ ]:
row = markets.row(144, named=True)
slug, start, end = row["market_slug"], row["start_time"].replace(tzinfo=None), row["end_time"].replace(tzinfo=None)
ev = l2.filter(pl.col("market_slug") == slug).collect()
window = ev.filter(pl.col("timestamp").is_between(start, end))
print(slug, "| outcome:", row["winning_outcome"], "|", window.group_by("event_type").len().sort("len", descending=True))

top = window.filter(pl.col("event_type") == "price_change")
plt.figure(figsize=(10, 4))
plt.step(top["timestamp"], top["best_bid"], where="post", lw=0.8, label="best bid (Up)")
plt.step(top["timestamp"], top["best_ask"], where="post", lw=0.8, label="best ask (Up)")
plt.title(f"{slug}: best bid and ask over the 5-minute window")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Full depth from the latest `book` snapshot before the middle of the window
mid = start + (end - start) / 2
snap = ev.filter((pl.col("event_type") == "book") & (pl.col("timestamp") <= mid)).tail(1)
book = snap.row(0, named=True)
print("snapshot at", book["timestamp"])
print(pl.DataFrame({
    "bid_price": book["bid_prices"][:5], "bid_size": book["bid_sizes"][:5],
    "ask_price": book["ask_prices"][:5], "ask_size": book["ask_sizes"][:5],
}))

## 4. Check outcomes against Chainlink TWAP
Up wins when the BTC/USD 60s TWAP at the end of the window is at least the TWAP at the start.
Prices are scaled by 10^18. Use the last report at or before each time.
From 2026-08-14 on, this rule matches 2,263 of 2,265 recorded outcomes; the first days after the 2026-08-07
switch to TWAP resolution match less often (94%).

In [ ]:
twap = (
    pl.read_parquet(f"{DATA}/chainlink_streams_twap60s/*/*.parquet")
    .select("observationsTimestamp", (pl.col("price").cast(pl.Float64) / 1e18).alias("twap"))
    .unique("observationsTimestamp", keep="last")
    .sort("observationsTimestamp")
)
m = markets.with_columns(
    pl.col("start_time").dt.replace_time_zone(None),
    pl.col("end_time").dt.replace_time_zone(None),
)
for col in ["start_time", "end_time"]:
    m = m.sort(col).join_asof(
        twap.rename({"observationsTimestamp": f"{col}_obs", "twap": f"{col}_twap"}),
        left_on=col, right_on=f"{col}_obs", strategy="backward",
    )
m = m.with_columns(
    pl.when(pl.col("end_time_twap") >= pl.col("start_time_twap")).then(pl.lit("yes")).otherwise(pl.lit("no")).alias("twap_outcome")
)
m = m.filter(pl.col("winning_outcome").is_not_null())
daily = m.group_by(pl.col("start_time").dt.date().alias("day")).agg(
    (pl.col("twap_outcome") == pl.col("winning_outcome")).mean().alias("match_rate"), pl.len().alias("markets")
).sort("day")
print(daily)

## More
- Every Polymarket market since 2026-05-01 and every crypto Up/Down series since 2026-02-01: [pmdata.dev](https://pmdata.dev) (free trial: 600 downloads).
- Fill-level backtests with queue position and latency: [pm-hftbacktest](https://github.com/mileswangs/pm-hftbacktest).